# VisionProf: các lần chạy Colab T4 cho giữa kỳ

Chạy **một lần từ trên xuống** (Runtime → Run all). Runtime phải là **T4 GPU**
(Runtime → Change runtime type → T4 GPU). Tổng thời gian khoảng 45–75 phút.

| Ô | Việc | Của ai | Ghi vào |
| :-: | :--- | :-: | :--- |
| 3 | TN3, TN4, TN5 trên T4 (đo lại thật, thay số liệu sinh ra đã xoá) | TV2 | `results/colab_t4/*/TN3, TN4, TN5` |
| 4 | TN6 phần nạp dữ liệu: DataLoader thật, `num_workers` 0 / 2 / 4 | TV3 | `results/colab_t4/*/TN6/bs32_fp32_w*` |
| 5 | Kiểm chứng code mức Layer đã sửa trên T4 (TN2) | TV1 | `/content/verify_results` (không ghi đè) |
| 6–7 | Tự kiểm tra + bảng tóm tắt | cả nhóm | |
| 8 | Nén kết quả và tải về | cả nhóm | `visionprof_colab_results.zip` |

Nếu Colab ngắt giữa chừng: chạy lại từ ô 1. Ô 3 có `--skip-existing` nên không đo lại những gì đã xong
(chỉ còn giữ được nếu runtime chưa bị xoá; xem ô 2b để lưu vào Google Drive).

In [ ]:
# Ô 1. Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0), "Hãy đổi runtime sang T4 GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Ô 2. Lấy code mới nhất và cài thư viện
%cd /content
!rm -rf VisionProf && git clone -q https://github.com/Huu-Tuan13-09/VisionProf.git
%cd /content/VisionProf
!git log -1 --format='%h %s'
!pip install -q -r requirements.txt

In [ ]:
# Ô 2b. Lưu kết quả mới thẳng vào Google Drive để không mất khi Colab ngắt.
# Đổi USE_DRIVE = True để bật. Các ô sau không cần số liệu T4 cũ (TN1, TN2) có trong bản clone.
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/VisionProf_results/colab_t4
    !rm -rf results/colab_t4 && ln -s /content/drive/MyDrive/VisionProf_results/colab_t4 results/colab_t4
    !ls -la results

In [ ]:
# Ô 3. TV2: TN3, TN4, TN5 trên Colab T4
import subprocess, time
MODELS = ["mobilenet_v3_large", "resnet50", "vit_b_16", "yolov8n", "distilbert", "bert_base", "mlp", "xgboost"]
TORCH_MODELS = [m for m in MODELS if m != "xgboost"]
NLP = {"distilbert", "bert_base"}
failed = []

def run(args):
    cmd = ["python", "scripts/run_experiment.py", "--device", "colab_t4", "--skip-existing", *args]
    print("▶", " ".join(cmd[2:]), flush=True)
    t = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        failed.append(" ".join(cmd)); print(r.stderr[-1500:])
    print(f"   {'OK' if r.returncode == 0 else 'LỖI'} · {time.time() - t:.0f}s", flush=True)

for m in MODELS:                                                    # TN3: batch 1, FP32
    run(["--model", m, "--experiment", "TN3", "--batch-size", "1", "--precision", "fp32", "--profilers", "layer"])
for m in MODELS:                                                    # TN4: batch 1/8/32 (+ seq 64/128/256 cho NLP)
    for bs in (1, 8, 32):
        for seq in ((64, 128, 256) if m in NLP else (128,)):
            run(["--model", m, "--experiment", "TN4", "--batch-size", str(bs), "--seq-len", str(seq), "--precision", "fp32"])
for m in TORCH_MODELS:                                              # TN5: FP16, batch 1
    run(["--model", m, "--experiment", "TN5", "--batch-size", "1", "--precision", "fp16"])
print("\nLệnh lỗi:", failed or "không có")

In [ ]:
# Ô 4. TV3: TN6 phần nạp dữ liệu (DataLoader thật, JPEG giải mã + augmentation trên CPU)
import os; print("Số vCPU của Colab:", os.cpu_count())
!python scripts/run_tn6_dataloader.py --device colab_t4 --batch-size 32 --workers 0 2 4

In [ ]:
# Ô 5. TV1: kiểm chứng code mức Layer đã sửa trên T4 (không ghi đè results/)
import os, pandas as pd
os.environ["VISIONPROF_RESULTS"] = "/content/verify_results"
!python experiments/tn2_validate_profiler.py --device cuda
del os.environ["VISIONPROF_RESULTS"]
cols = ["model", "overhead_pct", "layer_sum_ms", "t_base_ms", "total_rel_error_pct", "timer_floor_ms"]
old = pd.read_csv("results/analysis/TN2/summary_colab_t4.csv")[cols].assign(ban="báo cáo")
new = pd.read_csv("/content/verify_results/analysis/TN2/summary_colab_t4.csv")[cols].assign(ban="code đã sửa")
print(pd.concat([old, new]).sort_values(["model", "ban"]).round(3).to_string(index=False))

In [ ]:
# Ô 6. Tự kiểm tra: số liệu T4 là đo thật (phải in 0 thư mục) và đủ thư mục
import csv, glob, json
bad = []
for f in sorted(glob.glob("results/*/*/TN[345]/*/summary.json")):
    d = f.rsplit("/", 1)[0]
    s = json.load(open(f)); env = json.load(open(d + "/env.json"))
    p50 = s.get("p50_ms") or 0
    formula = p50 > 0 and abs(s["p90_ms"] - round(p50 * 1.05, 4)) < 1e-3 and abs(s["p99_ms"] - round(p50 * 1.15, 4)) < 1e-3
    ts = [float(r["timestamp"]) for r in csv.DictReader(open(d + "/benchmark.csv")) if r.get("timestamp")]
    if formula or "os" not in env or (ts and ts[0] == 1728000000.0):
        bad.append(d)
print(f"{len(bad)} thư mục có dấu hiệu số liệu sinh ra", *bad[:5], sep="\n  ")
for tn, expect in (("TN3", 8), ("TN4", 6 * 3 + 2 * 9), ("TN5", 7)):
    n = len(glob.glob(f"results/colab_t4/*/{tn}/*/summary.json"))
    print(f"{tn}: {n}/{expect} thư mục có summary.json", "✅" if n == expect else "⚠️ thiếu")
print("TN6 nạp dữ liệu:", len(glob.glob("results/colab_t4/*/TN6/*_w*/dataloader_summary.json")), "/ 6")

In [ ]:
import os
# Ô 7. Bảng tóm tắt để đưa vào slide
import glob, json, pandas as pd
rows = []
for f in glob.glob("results/colab_t4/*/TN3/bs1_fp32/summary.json"):
    s = json.load(open(f)); m = f.split("/")[2]
    cpu = f.replace("colab_t4", "cpu_laptop")
    c = json.load(open(cpu)) if os.path.exists(cpu) else {}
    rows.append({"model": m, "T4_p50_ms": s.get("p50_ms"), "T4_p99_ms": s.get("p99_ms"),
                 "T4_fps": s.get("throughput_fps"), "T4_vram_mb": s.get("peak_vram_mb"),
                 "CPU(Mac TV2)_p50_ms": c.get("p50_ms")})
print("TN3 (batch 1, FP32):"); print(pd.DataFrame(rows).sort_values("T4_p50_ms").round(2).to_string(index=False))
rows = []
for f in glob.glob("results/colab_t4/*/TN5/bs1_fp16/summary.json"):
    m = f.split("/")[2]; h = json.load(open(f)); f32 = f.replace("TN5/bs1_fp16", "TN3/bs1_fp32")
    if not os.path.exists(f32):
        print("Thiếu TN3 FP32 để so:", m); continue
    full = json.load(open(f32))
    rows.append({"model": m, "fp32_p50": full.get("p50_ms"), "fp16_p50": h.get("p50_ms"),
                 "speedup": round(full["p50_ms"] / h["p50_ms"], 2), "vram_fp32": full.get("peak_vram_mb"),
                 "vram_fp16": h.get("peak_vram_mb"), "cosine": h.get("extra", {}).get("cosine_similarity")})
print("\nTN5 (FP16 so với FP32; cosine = so output FP16 với FP32 cùng trọng số, cùng input.")
print("  cosine = nan nghĩa là không so được, ví dụ MobileNetV3 khởi tạo ngẫu nhiên cho output toàn 0):"); print(pd.DataFrame(rows).round(3).to_string(index=False))
rows = [json.load(open(f)) for f in sorted(glob.glob("results/colab_t4/*/TN6/*_w*/dataloader_summary.json"))]
print("\nTN6 nạp dữ liệu (batch 32):")
print(pd.DataFrame([{"model": r["model"], "workers": r["num_workers"], "step_ms": r["step_ms"],
                     "data_load_%": r["data_load_pct"], "img/s": r["images_per_s"]} for r in rows]).to_string(index=False))

In [ ]:
# Ô 8. Nén kết quả mới và tải về máy
!rm -f /content/visionprof_colab_results.zip
!zip -qr /content/visionprof_colab_results.zip results/colab_t4 -i '*/TN3/*' '*/TN4/*' '*/TN5/*' '*_w[0-9]*/*'
!cd /content && zip -qr /content/visionprof_colab_results.zip verify_results
!unzip -l /content/visionprof_colab_results.zip | tail -1
from google.colab import files
files.download("/content/visionprof_colab_results.zip")